# ER Challenge — Kaggle workhorse

Run order:
1. **Cell 1** — unpack the dataset + code (auto-finds the zips in your attached dataset)
2. **Cell 2** — install pinned dependencies (needs **Internet ON** in notebook settings)
3. **Cell 3** — environment check
4. **Cell 4** — training-pair generation (~3 h at 4 vCPUs; resumable)
5. **Cell 5** — package outputs for download

For the long run: **File → Save & Run All (Commit)** runs it headless for up to 12 h — you can close the browser.

In [ ]:
# Cell 1 — find and unpack the zips
import glob, os, zipfile

hits = {}
for name in ("colab_data.zip", "colab_code.zip"):
    h = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert h, f"{name} not found — attach the dataset: sidebar > Add Input > Your Datasets"
    hits[name] = h[0]
    print(name, "->", h[0])

os.makedirs("/kaggle/working/work", exist_ok=True)
for name, p in hits.items():
    with zipfile.ZipFile(p) as z:
        z.extractall("/kaggle/working/work")
print("Unpacked:", os.listdir("/kaggle/working/work"))

In [ ]:
# Cell 2 — pinned dependencies (requires Internet ON: Settings > Internet)
!pip install -q polars==1.44.2 xgboost==3.2.0 rapidfuzz unidecode psutil

In [ ]:
# Cell 3 — environment check (expect 4 CPUs, ~30 GB RAM)
import psutil, os
vm = psutil.virtual_memory()
print(f"RAM: {vm.total/1e9:.1f} GB | CPUs: {os.cpu_count()}")

In [ ]:
# Cell 4 — training-pair generation (~3 h; resumable via checkpoints)
# If it ever stops: re-run this cell — it resumes from done_US.txt / done_India.txt
%cd /kaggle/working/work/code/business_entity_resolution
!python src/gen_training_pairs.py \
    --data-dir /kaggle/working/work/student_resource/dataset/train \
    --out-dir /kaggle/working/cache/pairs \
    --n-entities 500000

In [ ]:
# Cell 5 — package outputs (appear in the notebook's Output tab for download)
!cd /kaggle/working && zip -qr pairs.zip cache/pairs && ls -lh pairs.zip
print("Download pairs.zip from the Output tab when the run finishes.")